# Moissonnage

**Auteur :** Francis Lareau  
**Date :** 08/10/2026

> Adaptation fidèle du document R Markdown à JupyterLab et Python. Le texte, la numérotation, l'ordre des opérations, les noms d'objets et la logique pédagogique du document d'origine sont conservés autant que possible. Seules les bibliothèques et les constructions propres à R sont remplacées par leurs équivalents Python.

# 1 Introduction

La collecte de données est l'étape au cours de laquelle on recueille les informations nécessaires à la recherche. Le moissonnage (*scraping* ou *harvesting*) est un moyen parmi d'autres de collecter des données, consistant à explorer des sites web via un script ou un programme afin d'en extraire le contenu. Les pages web renfermant les contenus recherchés sont téléchargées, formant ainsi un ensemble de données brutes qu'on peut éventuellement diviser en :

1) données (les textes étudiés) ;  
2) métadonnées (les informations utiles à propos des textes étudiés) ;  
3) informations non retenues (inutiles au regard des objectifs de recherche).

Plusieurs outils sont disponibles pour effectuer le moissonnage du web. Dans le cadre de cette introduction, nous utilisons les bibliothèques **requests**, **urllib.robotparser** et **time**, qui permettent un moissonnage responsable du web, c.-à-d. en suivant les principes de :

1) se présenter à l'hôte ;  
2) demander la permission ;  
3) opérer lentement ;  
4) sans redite (demander une seule fois).

Plus précisément, notre collecte de données impliquera :

1) l'interrogation de sites internet pour noter des liens URL (*Uniform Resource Locator*) pouvant mener à des pages web renfermant les informations recherchées, puis le téléchargement de ces pages web ;  
2) l'introduction dans la structure de données des documents HTML téléchargés à l'aide de la bibliothèque **lxml** ;  
3) l'introduction des documents PDF téléchargés dans la structure de données à l'aide de la bibliothèque **pypdf**.

Notons que certaines opérations formeront une séquence de sous-opérations. En R, le symbole `%>%` de **magrittr** articulait ces opérations. En Python, nous les écrivons successivement, généralement en réaffectant le résultat à un objet portant un nom explicite. Enfin, procédons à l'importation des bibliothèques nécessaires.

## 1.1 Librairies

In [ ]:
# 1 Installer les bibliothèques si nécessaire
# Cette commande doit être exécutée dans une cellule Jupyter.
%pip install pandas requests lxml pypdf

In [ ]:
# 2 Charger les bibliothèques
import time
from pathlib import Path
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from lxml import html
from pypdf import PdfReader

# 2 Collecte des données brutes

## 2.1 Structuration des données brutes

La classe **DataFrame** de la bibliothèque **pandas** fournit une structure de données sous forme de tabloïde. Une manière de structurer les contenus extraits du web consiste à insérer une nouvelle ligne dans le tabloïde à mesure que de nouveaux documents sont moissonnés, puis à classer les informations relatives à ces documents dans des catégories préalablement identifiées, qui correspondront à des colonnes distinctes.

Le corpus sélectionné pour notre démonstration est la revue en ligne *Argumentation et Analyse du Discours*, de sorte que nos données brutes seront constituées des articles de cette revue. Une page web au format HTML correspondant à chaque article sera téléchargée et l'adresse de cette page sera inscrite dans la colonne **HTML_URL**. De plus, une version PDF de l'article sera téléchargée et l'adresse correspondante à chaque PDF sera inscrite dans la colonne **PDF_URL**. Ultérieurement, les documents HTML et PDF seront intégrés à notre structure de données, respectivement dans les colonnes **HTML** et **PDF**.

### 2.1.1 Structure de données et de métadonnées

In [ ]:
# Créer un dataframe vide ####
dataframe = pd.DataFrame({
    "HTML": pd.Series(dtype="object"),
    "HTML_URL": pd.Series(dtype="object"),
    "PDF": pd.Series(dtype="object"),
    "PDF_URL": pd.Series(dtype="object")
})

Avant de procéder au moissonnage, spécifions un répertoire **html_download_path** où seront téléchargés les articles au format HTML, ainsi qu'un répertoire **pdf_download_path** où seront téléchargés les articles au format PDF.

### 2.1.2 Endroit pour le téléchargement

In [ ]:
# 1 Noter le chemin d'accès au dossier des données brutes ####
download_path = Path.cwd() / "donnee_brute"

# 2 Créer le dossier de données brutes s'il n'existe pas ####
download_path.mkdir(parents=True, exist_ok=True)

# 3 Noter le chemin d'accès au dossier des articles en format PDF ####
pdf_download_path = download_path / "PDF"

# 4 Créer le dossier du chemin d'accès s'il n'existe pas ####
pdf_download_path.mkdir(parents=True, exist_ok=True)

# 5 Noter le chemin d'accès au dossier des articles en format HTML ####
html_download_path = download_path / "HTML"

# 6 Créer le dossier du chemin d'accès s'il n'existe pas ####
html_download_path.mkdir(parents=True, exist_ok=True)

# 7 Noter un fichier log ####
file_log = download_path / "log.txt"

## 2.2 Moissonnage

Avant d'élaborer une stratégie de moissonnage, on peut visualiser les opérations que le chercheur effectuerait manuellement pour collecter ses données, en identifiant les formes opérationnelles répétitives susceptibles de faire l'objet de boucles récursives. Par exemple, la revue *Argumentation et Analyse du Discours* possède une page web principale répertoriant les pages web de tous les volumes, lesquelles donnent accès aux pages web des articles de chaque volume.

La page web principale constituera notre point de départ (`https://journals.openedition.org/aad/166`) puisque, à partir de celle-ci, il est possible de :

1) noter l'adresse URL de tous les volumes, puis récursivement accéder aux pages web de ces volumes ;  
2) noter l'adresse URL des articles de chaque volume, inscrire cette adresse à la colonne **HTML_URL**, puis récursivement accéder et télécharger la page web de ces articles ;  
3) noter l'URL de l'article au format PDF, l'inscrire dans la colonne **PDF_URL**, puis télécharger le PDF.

Notons que les pages web de la revue commencent par `https://journals.openedition.org/aad/`. Cette partie commune est notée dans **main_url**. En Python, la fonction **urljoin()** permet de formuler une adresse URL complète à partir de la partie générale et de la partie particulière. Enfin, un objet **requests.Session()** permet d'ouvrir une session que nous utiliserons pour gérer poliment nos transactions avec l'hôte.

### 2.2.1 Préparation

In [ ]:
# 1 Noter la partie générale des adresses URL ####
main_url = "https://journals.openedition.org/aad/"

# 2 Coller la partie générale à celle particulière de l'adresse de départ ####
starting_url = urljoin(main_url, "166")

# 3 Ouvrir une session de navigation avec l'hôte ####
session = requests.Session()

# 3.1 Se présenter ####
user_agent = "UMLP's Python Webscraping Tutorial"
session.headers.update({"User-Agent": user_agent})

# 3.2 Définir un temps d'attente entre les requêtes ####
delay = 1

# 3.3 Lire les permissions indiquées dans robots.txt ####
robots_url = "https://journals.openedition.org/robots.txt"
robots = RobotFileParser()
robots.set_url(robots_url)
robots.read()

Dans un premier temps, on associe la page web de départ à l'objet **session**, puis on extrait les attributs contenant les URL des volumes. Ensuite, on extrait le texte de l'attribut, puis on colle ce texte, la partie spécifique, dans la partie générale de l'URL.

Dans le script ci-dessous, l'opération de :

1) naviguer d'une page web vers une autre s'effectue via la méthode **session.get()** ;  
2) lire et structurer une page web s'effectue via la fonction **html.fromstring()** ;  
3) isoler un attribut dans la page web s'effectue via la méthode **xpath()** ;  
4) compléter une URL relative s'effectue via la fonction **urljoin()**.

### 2.2.2 Moissonnage des volumes

In [ ]:
# Fonction auxiliaire pour effectuer une requête polie ####
def polite_get(url):
    # 1 Demander la permission ####
    if not robots.can_fetch(user_agent, url):
        raise PermissionError(f"Le fichier robots.txt interdit le moissonnage de {url}")

    # 2 Opérer lentement ####
    time.sleep(delay)

    # 3 Télécharger la ressource ####
    response = session.get(url, timeout=30)
    response.raise_for_status()
    return response

In [ ]:
# Noter l'adresse URL de tous les volumes ####

# 1 Naviguer vers la page de départ ####
starting_response = polite_get(starting_url)

# 2 Lire la page web de départ ####
starting_page = html.fromstring(starting_response.content)

# 3 Extraire les attributs où se trouvent les URL ####
volume_hrefs = starting_page.xpath('//*[@id="main"]/ul/li/a/@href')
# volume_hrefs = starting_page.xpath('//ul[contains(@class, "collection")]/li/a/@href')  # variante ####

# 4 Extraire le texte de la partie particulière des URL ####
# Avec lxml, la méthode xpath() retourne déjà les valeurs textuelles des attributs href.

# 5 Coller la partie générale à celle particulière des URL ####
volume_urls = [urljoin(starting_url, href) for href in volume_hrefs]

Dans un deuxième temps, on extrait l'URL des articles de chaque volume et on la note dans la colonne **HTML_URL**. Ensuite, on navigue vers chaque page web correspondant à un article pour extraire l'URL du PDF, que l'on notera dans la colonne **PDF_URL**. Enfin, la page web HTML et le PDF sont téléchargés avec **session.get()** et nommés en fonction du numéro de ligne où l'on se trouve dans la structure de données.

### 2.2.3 Moissonnage des articles

In [ ]:
# Boucle pour traiter chaque URL des volumes ####
# Note : on ne traite que l'avant-dernier volume pour sauver du temps ####
for volume_url in volume_urls[-2:-1]:

    # 1 Naviguer vers la page du volume ####
    volume_response = polite_get(volume_url)
    volume_page = html.fromstring(volume_response.content)

    # 2 Noter l'adresse URL de tous les articles ####

    # 2.1 Lire la page web du volume ####
    # La page a déjà été lue et structurée dans volume_page.

    # 2.2 Extraire les attributs où se trouvent les URL d'articles ####
    article_hrefs = volume_page.xpath('//div[contains(@class, "title")]/a/@href')

    # 2.3 Extraire le texte de la partie particulière des URL ####
    # xpath() retourne déjà les valeurs textuelles des attributs href.

    # 2.4 Coller la partie générale à celle particulière des URL ####
    article_urls = [urljoin(volume_url, href) for href in article_hrefs]

    # 3 Boucle pour traiter chaque URL des articles ####
    for article_url in article_urls:

        # 3.1 Naviguer vers la page web de l'article ####
        article_response = polite_get(article_url)
        article_page = html.fromstring(article_response.content)

        # 3.2 Noter l'adresse URL du PDF ####
        pdf_hrefs = article_page.xpath('//*[@id="wDownload"]/@href')
        pdf_url = urljoin(article_url, pdf_hrefs[0]) if pdf_hrefs else None

        # 3.3 Créer un nouveau dataframe (ancienne + nouvelle ligne) ####
        new_row = pd.DataFrame([{
            "HTML": None,
            # 3.3.1 Inscrire URL de l'article HTML ####
            "HTML_URL": article_url,
            "PDF": None,
            # 3.3.2 Inscrire URL de l'article PDF ####
            "PDF_URL": pdf_url
        }])
        dataframe = pd.concat([dataframe, new_row], ignore_index=True)

        # Numéro de la ligne courante, en commençant à 1 comme dans R ####
        row_id = len(dataframe)

        # 3.4 Télécharger l'article en format HTML ####
        # 3.4.1 Déterminer le nom du fichier ####
        html_destfile = f"{row_id}.html"
        # 3.4.2 Déterminer le chemin du dossier ####
        html_file = html_download_path / html_destfile
        html_file.write_bytes(article_response.content)

        # 3.5 Télécharger l'article en format PDF ####
        if pdf_url is not None:
            pdf_response = polite_get(pdf_url)
            pdf_destfile = f"{row_id}.pdf"
            pdf_file = pdf_download_path / pdf_destfile
            pdf_file.write_bytes(pdf_response.content)

        # 3.6 Ajouter une entrée au log ####
        with file_log.open("a", encoding="utf-8") as log:
            log.write(f"{pd.Timestamp.now()} - Scraping {article_url}\n")

# 3 Importation des documents HTML

L'introduction des documents HTML dans la structure de données implique leur importation et leur conversion en chaînes de caractères. Nous effectuerons ces opérations à l'aide de la méthode **Path.read_text()**.

### 3.1 Importation des documents HTML

In [ ]:
# Inscrire l'article brut à la colonne 'HTML' ####
dataframe["HTML"] = [
    # 1 Énumération des numéros de ligne du dataframe ####
    # 2 Construire le chemin d'accès de l'article ####
    # 3 Lire/décoder l'article en format HTML ####
    # 4 Transformer l'article en chaîne de caractères ####
    (html_download_path / f"{row_id}.html").read_text(
        encoding="utf-8",
        errors="replace"
    )
    for row_id in range(1, len(dataframe) + 1)
]

Note : une compréhension de liste Python permet d'appliquer la même opération à tous les éléments d'une séquence et de produire une liste de résultats de même dimension. Elle joue ici le même rôle que la succession de **lapply()** et **sapply()** dans le code R d'origine.

# 4 Importation des documents PDF

L'introduction des documents PDF dans la structure de données implique leur importation et leur conversion en chaînes de caractères, que nous effectuerons à l'aide de la bibliothèque **pypdf**. La classe qui nous intéresse est **PdfReader**, qui permet d'extraire le contenu textuel de chaque page du PDF. Nous utiliserons ensuite la méthode **join()** afin de joindre ces pages en une seule chaîne de caractères correspondant ici à un article, en insérant un indicateur de changement de page contenant le numéro de la page (`<page_n>`).

Dans l'exemple suivant, une compréhension de liste assemble chaque page du PDF avec son numéro de page, puis **join()** colle les éléments résultants afin de ne former qu'une seule chaîne de caractères.

### 4.1 Importation des documents PDF

In [ ]:
# Fonction permettant de lire un PDF et d'assembler ses pages ####
def pdf_text(pdf_path):
    reader = PdfReader(pdf_path)
    pages = [page.extract_text() or "" for page in reader.pages]

    return "".join(
        f"{page_text}\n<page_{page_number}>\n"
        for page_number, page_text in enumerate(pages, start=1)
    )

In [ ]:
# Inscrire l'article à la colonne 'PDF' ####
dataframe["PDF"] = [
    # 1 Énumération des numéros de ligne du dataframe ####
    # 2 Construire le chemin d'accès de l'article ####
    # 3 Lire/décoder l'article en format PDF ####
    # 4 Transformer l'article en une seule chaîne de caractères ####
    # 5 Ajouter les caractères manquants à la fin de l'article ####
    pdf_text(pdf_download_path / f"{row_id}.pdf")
    if (pdf_download_path / f"{row_id}.pdf").exists()
    else None
    for row_id in range(1, len(dataframe) + 1)
]

# 5 Sauvegarde de la structure de données

La méthode **to_pickle()** sauvegarde un seul objet Python dans un fichier compressible qui conserve intégralement la structure du DataFrame, contrairement aux formats CSV ou XLS. Elle constitue ici l'équivalent fonctionnel le plus proche de **saveRDS()**.

### 5.1 Sauvegarde de la structure de données

In [ ]:
# Sauvegarder le dataframe au format pickle ####
dataframe.to_pickle(download_path / "01_moissonage_exemple_dataframe.pkl")

# 6 Exercices

L'objectif de la série d'exercices est de moissonner la *Revue d'Anthropologie des Connaissances* disponible en ligne sur OpenEdition (`https://journals.openedition.org/rac/263`). Pour ce faire, nous devons identifier des URL à l'aide de requêtes XPath. Notons que ces adresses ou hyperliens correspondent généralement à des éléments dont l'attribut **href** spécifie l'URL. Directement dans le navigateur, on peut inspecter un hyperlien en faisant un clic droit sur l'élément qui nous intéresse, puis en cliquant sur *Inspecter*.

## 6.1 Le moissonnage en pièces détachées

Débutons par l'ouverture d'une session avec **requests.Session()**. Exécutez le code suivant après avoir déterminé la partie générale et la partie particulière de la page de départ.

### 6.1.1 Préparation

In [ ]:
main_url = "..."  # Veuillez remplacer ... par la partie générale de l'URL ####

starting_url = urljoin(main_url, "...")  # Veuillez remplacer ... par la partie particulière de l'URL ####

session = requests.Session()
user_agent = "UMLP's Python Webscraping Tutorial"
session.headers.update({"User-Agent": user_agent})
delay = 1

Nous allons extraire l'URL de tous les numéros présents sur la page web de départ contenant ces informations. Veuillez compléter le script suivant en inscrivant la formule XPath.

### 6.1.2 URL des numéros

In [ ]:
starting_response = polite_get(starting_url)
starting_page = html.fromstring(starting_response.content)

volume_hrefs = starting_page.xpath("...")  # Veuillez remplacer ... par la formule xpath ####
volume_urls = [urljoin(starting_url, href) for href in volume_hrefs]

Nous allons extraire les URL de tous les articles d'un des volumes précédemment identifiés (`https://journals.openedition.org/rac/2162`). Veuillez compléter le script suivant en inscrivant la formule XPath.

### 6.1.3 URL des articles

In [ ]:
volume_url = "https://journals.openedition.org/rac/2162"

volume_response = polite_get(volume_url)
volume_page = html.fromstring(volume_response.content)

article_hrefs = volume_page.xpath("...")  # Veuillez remplacer ... par la formule xpath ####
article_urls = [urljoin(volume_url, href) for href in article_hrefs]

Nous allons télécharger la page web d'un des articles. Veuillez compléter le script suivant de manière à enregistrer la page de l'article dans un fichier nommé **1_scraping_exercice.html** et situé dans le répertoire de travail.

### 6.1.4 Page web d'un article en format HTML

In [ ]:
article_url = article_urls[0]
article_response = polite_get(article_url)

# Veuillez compléter le code ####

## 6.2 Le moissonnage d'un volume de la *Revue d'Anthropologie des Connaissances*

En vous inspirant des étapes précédentes, inscrivez les URL de tous les articles d'un même volume du journal dans des cellules distinctes de la structure de données **df** et téléchargez les pages web de ces articles. Veuillez compléter la partie du code permettant de produire les objets **article_urls**, les valeurs **df.loc[row_id, "HTML_URL"]** et les documents HTML qui devront porter le nom de la ligne correspondante dans la structure de données et se trouver dans un répertoire nommé **1_HTML_exercice**.

### 6.2.1 Moissonnage

In [ ]:
df = pd.DataFrame({
    "HTML": pd.Series(dtype="object"),
    "HTML_URL": pd.Series(dtype="object")
})

html_download_path = Path.cwd() / "1_HTML_exercice"
html_download_path.mkdir(parents=True, exist_ok=True)

article_hrefs = volume_page.xpath("...")  # Veuillez remplacer ... par la formule xpath ####
article_urls = [urljoin(volume_url, href) for href in article_hrefs]

for article_url in article_urls:
    article_response = polite_get(article_url)

    # Veuillez compléter le code permettant d'ajouter la ligne à df ####

    # Veuillez compléter le code permettant de télécharger le fichier HTML ####

En vous inspirant du script de la section [3 Importation des documents HTML], veuillez importer les documents HTML précédemment téléchargés dans la structure de données. Plus précisément, veuillez inscrire les documents HTML décodés en chaînes de caractères dans la colonne **HTML** de la structure de données **df**.

### 6.2.2 Importation des documents HTML

In [ ]:
# Veuillez inscrire le code ####

## 6.3 Sauvegarde de la structure de données

Veuillez enregistrer la structure de données résultante **df** dans un fichier pickle nommé **1_scraping_dataframe_exercice.pkl**.

### 6.3.1 Sauvegarde

In [ ]:
# Veuillez inscrire le code ####